# 03. Eine Filmreview mit N-Grammen erzeugen

**Leitfrage:** Wie verändert sich ein erzeugter Text, wenn das Modell die letzten zwei Wörter kennt?

**Arbeitsauftrag:** [Ein Folgewort gewichtet ziehen](#sampling) und [drei Modelle ausprobieren](#generate). Du brauchst Listen, Wörterbücher und eine `for`-Schleife. Alle Daten liegen als TXT-Dateien bei.

## 1. Wörter nach `ending` zählen

Lies die drei kurzen Filmurteile. Welches Wort folgt zweimal auf `ending`? Welches folgt einmal?

In [ ]:
example_sentences = [
    ['the', 'ending', 'works'],
    ['the', 'ending', 'drags'],
    ['this', 'ending', 'works'],
]

`count_followers(sentences, n)` erhält eine Liste von Sätzen. Jeder Satz ist bereits eine Liste von Tokens. Bei `n=2` besteht der Kontext aus einem Wort. Das Ergebnis ist ein Wörterbuch: Zum Schlüssel `('ending',)` gehört ein weiteres Wörterbuch mit den **Folgewörtern und ihren Häufigkeiten**. Das Komma kennzeichnet ein Tupel mit einem Eintrag. [Code zum Nachlesen](ngram_data.py)

In [ ]:
from ngram_data import count_followers, load_sentences, next_word_counts, tokenize

example_counts = count_followers(example_sentences, n=2)
ending_counts = example_counts[('ending',)]
print(ending_counts)
print('Probability of works:', ending_counts['works'] / sum(ending_counts.values()))

## 2. Was folgt auf `the special`?

Wir zählen in den **500 Trainingsreviews**. Der Lader trennt an Punkt, Ausrufe- und Fragezeichen und liest englische Wörter in Kleinschreibung. Satzgrenzen bleiben erhalten. `<start>` dient als Kontext am Anfang, `<end>` kann als nächstes Token gezogen werden. Die Sentiment-Labels spielen hier keine Rolle.

In [ ]:
from pathlib import Path

training_sentences = load_sentences(Path('data/imdb/train'))
models = {}
for n in (1, 2, 3):
    models[n] = count_followers(training_sentences, n)
print('Training sentences:', len(training_sentences))

Was ändert sich für `effects`? Das Unigramm-Modell zählt alle Ziel-Tokens (`()` ist der leere Kontext). Das Bigramm-Modell zählt nur Fortsetzungen von `special`, das Trigramm-Modell nur Fortsetzungen von `the special`.

In [ ]:
contexts = [(), ('special',), ('the', 'special')]
for n, context in enumerate(contexts, start=1):
    counts = models[n][context]
    matches = counts['effects']
    total = sum(counts.values())
    print(f'{n}-gram: {matches} / {total} = {matches / total:.2%}')

<a id="sampling"></a>
## 3. Einen Reviewtext erzeugen

### 3.1 Ein Folgewort gewichtet ziehen

**Dein Auftrag:** Ergänze die Rückgabe in `sample_next_word`. Bei zwei Belegen für `works` und einem für `drags` soll `works` doppelt so wahrscheinlich sein. `rng.choices` akzeptiert Wörter und Gewichte in derselben Reihenfolge. Auch bei einer Ziehung liefert die Methode eine Liste. Gib das gezogene **Wort** zurück. [Python-Dokumentation zu `choices`](https://docs.python.org/3/library/random.html#random.choices)

In [ ]:
from random import Random


def sample_next_word(counts: dict[str, int], rng: Random) -> str:
    words = list(counts)
    weights = list(counts.values())
    # TODO: Draw one word using the counts as weights.
    raise NotImplementedError

In [ ]:
rng = Random(42)
draws = [sample_next_word(ending_counts, rng) for _ in range(600)]
print('works:', draws.count('works'), '| drags:', draws.count('drags'))
assert 350 < draws.count('works') < 450

<a id="generate"></a>
### 3.2 Drei Modelle ausprobieren

Nach `the special` zieht das Trigramm-Modell mit Wahrscheinlichkeit 11/13 das Wort `effects`. Der Kontext ist jetzt `special effects`. Darauf folgt mit Wahrscheinlichkeit 6/22 das Wort `are`. Laut Kettenregel ist die Chance für genau diese beiden Ziehungen (11/13) · (6/22), also rund 23 %. Bei einem unbekannten Satzanfang verkürzt der Generator den Kontext von links, bis er Fortsetzungen kennt. Bei `<end>` hört er auf, spätestens nach der vorgegebenen Zahl neuer Wörter.

In [ ]:
def generate_review(start: str, n: int, seed: int, max_new_words: int = 7) -> str:
    rng = Random(seed)
    words = tokenize(start)
    for _ in range(max_new_words):
        counts = next_word_counts(models, words, n)
        word = sample_next_word(counts, rng)
        if word == '<end>':
            break
        words.append(word)
    return ' '.join(words)

In [ ]:
start = 'the special'
seed = 405
for n in (1, 2, 3):
    print(f'{n}-gram: {generate_review(start, n, seed)}')

Probiere `the ending` oder einen eigenen Anfang aus. Halte den Startwert zunächst gleich, damit der Vergleich wiederholbar ist. Ändere dann nur `seed`. Welche Wortfolge passt lokal gut, und an welcher Stelle verliert der Text seinen Sinn? [Herkunft der IMDb-Kursdaten](data/imdb/README.md)